### Quiz( data/wine-data.csv )
* 각 분류기들을 이용하여 모델을 생성하고 평가하시오
* class : 0 - 레드와인, 1 - 화이트와인

### 데이터셋 로드

In [9]:
import pandas as pd
wine = pd.read_csv('../data_set/5.스케일링/wine-data.csv')
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0
1,9.8,2.6,3.20,0
2,9.8,2.3,3.26,0
3,9.8,1.9,3.16,0
4,9.4,1.9,3.51,0


In [10]:
wine.columns

Index(['alcohol', 'sugar', 'pH', 'class'], dtype='object')

In [11]:
features = ['alcohol','sugar','pH']
label = 'class'
X,y = wine[features], wine[label]

### train, test 데이터셋 구분

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=62)

### KNN

In [13]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings('ignore')

#### KNN 예측

In [14]:
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
print('train :',knn.score(X_train, y_train))
print('train :',knn.score(X_test, y_test))

train : 0.9091783721377718
train : 0.8415384615384616


#### MinMax 스케일링

In [15]:
minMaxScaler = MinMaxScaler()
minMaxScaler.fit(X_train)
X_train_minMax = minMaxScaler.transform(X_train)
X_test_minMax = minMaxScaler.transform(X_test)

In [16]:
knn = KNeighborsClassifier()
knn.fit(X_train_minMax, y_train)
print('train :',knn.score(X_train_minMax, y_train))
print('train :',knn.score(X_test_minMax, y_test))

train : 0.8935924571868386
train : 0.8307692307692308


#### 하이퍼파라미터 및 GridSearchCV

In [17]:
from sklearn.model_selection import GridSearchCV
params = {
    'n_neighbors': range(1,11),#[1], 1으로 설정하면 과대적합이 나온다는 것을 보여주자
    'metric' : ['manhattan','euclidean' ], # 거리기반 알고리즘
    'weights' : ['uniform','distance'] # 균등가중치, 가까운 데이터 가중치
}
knn = KNeighborsClassifier() #모델 생성

grid_cv = GridSearchCV(knn, param_grid = params, cv=3, n_jobs = -1)
grid_cv.fit(X_train_minMax, y_train)

print('최적의 하이퍼 파라미터 : ', grid_cv.best_params_)

print(grid_cv.score(X_train_minMax, y_train))
print(grid_cv.score(X_test_minMax, y_test))

최적의 하이퍼 파라미터 :  {'metric': 'manhattan', 'n_neighbors': 10, 'weights': 'distance'}
0.9976909755628247
0.8615384615384616


### svm

#### 선형 모델

In [18]:
from sklearn.svm import SVC

svc_linear = SVC(kernel = 'linear')
svc_linear.fit(X_train, y_train)
print("train : ",svc_linear.score(X_train,y_train))
print("test : ",svc_linear.score(X_test,y_test))

train :  0.7821820280931306
test :  0.7884615384615384


#### StandardScaler
* MinMax보다 예측력이 좋아서 Standard를 사용했다

In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaler = scaler.transform(X_train)
X_test_scaler = scaler.transform(X_test)

In [20]:
svc_linear.fit(X_train_scaler, y_train)

print("train : ",svc_linear.score(X_train_scaler,y_train))
print("test : ",svc_linear.score(X_test_scaler,y_test))

train :  0.7837213777179142
test :  0.7876923076923077


#### 하이퍼파라미터 및 GridSearchCV

In [21]:
#GridSearchCV
from sklearn.model_selection import GridSearchCV

param_range = [ 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
params = {
    'C':param_range
}

svc = SVC(kernel = 'linear',)

grid_cv = GridSearchCV(svc, param_grid = params, cv=3, n_jobs = -1)# n_jobs : 사용할 코어의 수 . -1인경우 모든 코어를 사용함
grid_cv.fit(X_train_scaler, y_train)

print('최적의 하이퍼 파라미터 : ', grid_cv.best_params_)

print('train :',grid_cv.score(X_train_scaler,y_train))
print('test :',grid_cv.score(X_test_scaler,y_test))

최적의 하이퍼 파라미터 :  {'C': 10.0}
train : 0.78410621512411
test : 0.7853846153846153


#### 비선형 모델

In [22]:
svc_rbf = SVC(kernel = 'rbf')
svc_rbf.fit(X_train, y_train)
print("train : ",svc_rbf.score(X_train,y_train))
print("test : ",svc_rbf.score(X_test,y_test))

train :  0.8526072734269771
test :  0.8423076923076923


#### StandardScaler

In [23]:
# 비선형모델 스케일링
svc_rbf.fit(X_train_scaler, y_train)

print("train : ",svc_rbf.score(X_train_scaler,y_train))
print("test : ",svc_rbf.score(X_test_scaler,y_test))

train :  0.8318260534923995
test :  0.81


#### 하이퍼파라미터 및 GridSearchCV

In [24]:
#GridSearchCV
from sklearn.model_selection import GridSearchCV

param_c = [ 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
param_gamma = [ 0.001, 0.01, 0.1, 1.0, 10.0 ]
params = {
    'C':param_c,
    'gamma':param_gamma
}

svc = SVC(kernel = 'rbf')

grid_cv = GridSearchCV(svc, param_grid = params, cv=3, n_jobs = -1)# n_jobs : 사용할 코어의 수 . -1인경우 모든 코어를 사용함
grid_cv.fit(X_train_scaler, y_train)
print('최적의 하이퍼 파라미터 : ', grid_cv.best_params_)

print('train :',grid_cv.score(X_train_scaler,y_train))
print('test :',grid_cv.score(X_test_scaler,y_test))

최적의 하이퍼 파라미터 :  {'C': 100.0, 'gamma': 1.0}
train : 0.8858957090629209
test : 0.8438461538461538


### RandomForest

In [25]:
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier()

rfc.fit(X_train, y_train)

print("train : ",rfc.score(X_train,y_train))
print("test : ",rfc.score(X_test,y_test))

train :  0.9976909755628247
test :  0.8853846153846154


#### 하이퍼파라미터 및 GridSearchCV

In [26]:
#GridSearchCV
params = {
    'n_estimators':range(10,101,10),
    'max_depth' : range(10,15,2),
    'min_samples_leaf' : range(5,21,5),
    'min_samples_split' : range(4,21,4)
}

rfc = RandomForestClassifier()

grid_cv = GridSearchCV(rfc, param_grid = params, cv=3, n_jobs = -1)# n_jobs : 사용할 코어의 수 . -1인경우 모든 코어를 사용함
grid_cv.fit(X_train, y_train)

print('최적의 하이퍼 파라미터 : ', grid_cv.best_params_)

print("train : ",grid_cv.score(X_train,y_train))
print("test : ",grid_cv.score(X_test,y_test))

최적의 하이퍼 파라미터 :  {'max_depth': 14, 'min_samples_leaf': 5, 'min_samples_split': 4, 'n_estimators': 70}
train :  0.9226476813546277
test :  0.8684615384615385


### GradientBoosting

In [27]:
from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)

print("train : ",gbc.score(X_train,y_train))
print("test : ",gbc.score(X_test,y_test))

train :  0.8862805464691168
test :  0.8492307692307692


#### 하이퍼파라미터 및 GridSearchCV

In [28]:
import numpy as np
from sklearn.model_selection import GridSearchCV
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=62)
params = {
    'learning_rate':[0.1,0.3,0.5,0.7,0.9],
    'n_estimators' : [100,300,500,700],
    'subsample' : np.arange(0.1, 1, 0.2) #실수값은 range로 안된다.
}

gbc = GradientBoostingClassifier()

grid_cv = GridSearchCV(gbc, param_grid = params, cv=3, n_jobs = -1)# n_jobs : 사용할 코어의 수 . -1인경우 모든 코어를 사용함
grid_cv.fit(X_train, y_train)
print('최적의 하이퍼 파라미터 : ', grid_cv.best_params_)

print("train : ",grid_cv.score(X_train,y_train))
print("test : ",grid_cv.score(X_test,y_test))

최적의 하이퍼 파라미터 :  {'learning_rate': 0.1, 'n_estimators': 700, 'subsample': np.float64(0.9000000000000001)}
train :  0.9303444294785453
test :  0.8623076923076923
